# Langkah 1 — Pipeline Data & Training Loop
**Proyek:** LeNet · AlexNet · VGG · GoogLeNet pada CIFAR-10

Notebook ini adalah *infrastruktur bersama*. Keempat model nanti memakai **loader, loop training, dan metrik yang sama** supaya perbandingannya adil.

> Runtime → Change runtime type → **T4 GPU** sebelum menjalankan.

In [ ]:
import math, time
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, models

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Aktifkan GPU di Colab: Runtime -> Change runtime type -> T4 GPU"
torch.manual_seed(42)
print(device, torch.cuda.get_device_name(0))

## 1. Muat CIFAR-10 langsung ke GPU
Colab hanya punya ~2 core CPU. Kalau resize 32→224 dilakukan di CPU (pakai `transforms.Resize`), GPU akan banyak menganggur menunggu data.
Solusinya: simpan seluruh dataset sebagai tensor `uint8` di GPU (~150 MB), lalu augmentasi & resize dilakukan di GPU per batch.

In [ ]:
def load_cifar10(train):
    ds = datasets.CIFAR10("./data", train=train, download=True)
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous()  # (N, 3, 32, 32), uint8
    y = torch.tensor(ds.targets)
    return x, y

x_full, y_full = load_cifar10(train=True)
x_test, y_test = load_cifar10(train=False)

# Split validasi 5.000 citra. Test set HANYA dipakai sekali di akhir.
perm = torch.randperm(len(x_full), generator=torch.Generator().manual_seed(42))
x_val,   y_val   = x_full[perm[:5000]], y_full[perm[:5000]]
x_train, y_train = x_full[perm[5000:]], y_full[perm[5000:]]

CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
print(x_train.shape, x_val.shape, x_test.shape)

## 2. Loader di GPU (augmentasi + resize + normalisasi)
Dua konfigurasi yang akan kita pakai:

| Model | Bobot | `size` | mean/std |
|---|---|---|---|
| LeNet | dari nol | 32 | statistik CIFAR-10 |
| AlexNet, VGG, GoogLeNet | pretrained ImageNet | 224 | statistik **ImageNet** |

Model pretrained harus melihat input dengan resolusi & distribusi yang mirip dengan saat ia dilatih.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
CIFAR_MEAN    = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD     = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

def augment(x):
    """Random horizontal flip (per citra) + random crop 32 dengan padding 4.
    Penyederhanaan: offset crop sama untuk satu batch (cepat, cukup untuk tugas ini)."""
    flip = torch.rand(x.size(0), device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i, j = torch.randint(0, 9, (2,)).tolist()
    return x[:, :, i:i + 32, j:j + 32]

class GPULoader:
    def __init__(self, x, y, batch_size, size, mean, std, train):
        self.x, self.y = x.to(device), y.to(device)
        self.bs, self.size, self.train = batch_size, size, train
        self.mean, self.std = mean.to(device), std.to(device)

    def __len__(self):
        return math.ceil(len(self.x) / self.bs)

    def __iter__(self):
        n = len(self.x)
        idx = torch.randperm(n, device=device) if self.train else torch.arange(n, device=device)
        for k in range(0, n, self.bs):
            b = idx[k:k + self.bs]
            x = self.x[b].float() / 255.0
            if self.train:
                x = augment(x)
            if self.size != 32:
                x = F.interpolate(x, size=self.size, mode="bilinear", align_corners=False)
            yield (x - self.mean) / self.std, self.y[b]

def make_loaders(size, mean, std, batch_size=128):
    return (GPULoader(x_train, y_train, batch_size, size, mean, std, train=True),
            GPULoader(x_val,   y_val,   256,        size, mean, std, train=False),
            GPULoader(x_test,  y_test,  256,        size, mean, std, train=False))

## 3. Training loop generik (dengan mixed precision / AMP)

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        with torch.autocast(device, dtype=torch.float16):
            out = model(x)
        loss_sum += F.cross_entropy(out.float(), y, reduction="sum").item()
        correct  += (out.argmax(1) == y).sum().item()
        total    += y.numel()
    return loss_sum / total, correct / total

def fit(model, train_loader, val_loader, epochs, lr, weight_decay=5e-4):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]   # hormati layer yang di-freeze
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_loader))
    scaler = torch.amp.GradScaler(device)
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        t0, loss_sum, correct, total = time.time(), 0.0, 0, 0
        for x, y in train_loader:
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device, dtype=torch.float16):
                out = model(x)
                loss = F.cross_entropy(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.item() * y.numel()
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.numel()
        val_loss, val_acc = evaluate(model, val_loader)
        rec = dict(epoch=ep, train_loss=loss_sum / total, train_acc=correct / total,
                   val_loss=val_loss, val_acc=val_acc, sec=time.time() - t0)
        history.append(rec)
        print(f"ep {ep:2d} | train {rec['train_loss']:.3f} / {rec['train_acc']:.3f} "
              f"| val {val_loss:.3f} / {val_acc:.3f} | {rec['sec']:.0f}s")
    return history

## 4. Sanity check pipeline
Sebelum memakai model besar, buktikan dulu pipeline-nya benar dengan model paling sederhana: **regresi logistik** (Flatten + Linear).
Ekspektasi kasar: val acc **~35–40%** setelah beberapa epoch. Jauh di bawah itu (~10% = tebakan acak) berarti ada bug di data/label/normalisasi.

In [ ]:
train_dl, val_dl, test_dl = make_loaders(32, CIFAR_MEAN, CIFAR_STD)

x, y = next(iter(train_dl))
print("batch:", x.shape, x.dtype, "| mean≈0?", round(x.mean().item(), 3), "| std≈1?", round(x.std().item(), 3))
print("label contoh:", [CLASSES[i] for i in y[:8].tolist()])

baseline = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10))
hist = fit(baseline, train_dl, val_dl, epochs=5, lr=0.01)

## 5. TUGASMU — LeNet-5 (dari nol, input 32×32×3)
Spesifikasi (LeNet-5 yang diadaptasi ke citra RGB):

| Layer | Konfigurasi |
|---|---|
| C1 | Conv 3→6, kernel 5×5, tanpa padding |
| S2 | Pooling 2×2, stride 2 |
| C3 | Conv 6→16, kernel 5×5 |
| S4 | Pooling 2×2, stride 2 |
| F5 | FC → 120 |
| F6 | FC 120 → 84 |
| Out | FC 84 → 10 |

**Sebelum menulis kode, kerjakan di kertas:**
1. Ukuran output setiap layer — pakai ⌊(W − K + 2P)/S⌋ + 1. Berapa input F5?
2. Jumlah parameter setiap layer (termasuk bias) dan totalnya.
3. Keputusan desain: versi **asli** (Tanh + AvgPool) atau **modern** (ReLU + MaxPool)? Tulis prediksimu mana yang lebih akurat dan *kenapa* (petunjuk: slide 9).

Lalu verifikasi jawabanmu dengan `count_params` dan sebuah forward pass dummy.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: definisikan self.features (conv + aktivasi + pooling)
        # TODO: definisikan self.classifier (FC layers)
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

# Verifikasi:
# m = LeNet5()
# print(m(torch.randn(2, 3, 32, 32)).shape)   # harus torch.Size([2, 10])
# print(count_params(m))                       # bandingkan dengan hitunganmu
# hist_lenet = fit(m, train_dl, val_dl, epochs=15, lr=0.05)